# Operativos macro: productividad y tipo de participante

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `Operativos_Macro_-_SIGES.pdf` (y, para la verificación final, `261106_operativos_macro_v2.html`).

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdfplumber", "openpyxl"], check=False)

DATA_FILES = ["Operativos_Macro_-_SIGES.pdf"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

Contexto
Productividad de los operativos macro — Secretaría de Seguridad (DMQ)
Fuente: "Operativos Macro - SIGES" (170 operativos, con personal movilizado y resultados).
La productividad se mide como resultados por personal (u horas-persona) movilizado.

In [ ]:
# Parámetros y librerías
import re, json, csv, datetime, collections, unicodedata
import pdfplumber
try:
    MACRO_PDF = DATA_FILES[0]
except NameError:
    MACRO_PDF = "Operativos_Macro_-_SIGES.pdf"
    OUT = "salida"
    import os; os.makedirs(OUT, exist_ok=True)

def strip_acc(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

def norm(s):
    return strip_acc(s).upper().strip()

def dur(h):
    """Duración en horas de 'HH:MM – HH:MM' (cruza medianoche). None si el dato es inválido."""
    m = re.match(r'\s*(\d\d):(\d\d)\s*[–-]\s*(\d\d):(\d\d)', h or '')
    if not m:
        return None
    a = int(m[1]) * 60 + int(m[2]); b = int(m[3]) * 60 + int(m[4])
    d = (b - a) % 1440
    if a == b:
        return None            # '16:18 - 16:18': horario inválido, no se asume 24 h
    return d / 60

def start_hour(h):
    m = re.match(r'\s*(\d\d):(\d\d)', h or '')
    return int(m[1]) if m else None

INST_MAP = {
    'CACM': 'CACMQ', 'CACMQ': 'CACMQ', 'COMISAR A': 'Comisaría', 'COMISARÍA': 'Comisaría',
    'Q TURISMO': 'Quito Turismo', 'Q.TURISMO': 'Quito Turismo', 'QUITO TURISMO': 'Quito Turismo',
    'MIN. INTERIOR': 'Min. Interior', 'MIN INTERIOR': 'Min. Interior', 'MINISTERIO DEL INTERIOR': 'Min. Interior',
    'PPNN (MIGRACIÓN)': 'Migración', 'PPNN MIGRACIÓN': 'Migración', 'MIGRACIÓN': 'Migración', 'DIR. MIGRACIÓN': 'Migración',
    '(CAN MARCADOR)': 'CACMQ', 'POLICÍA (COSP)': 'PPNN', 'METROQ': 'Metro de Quito',
    'EPMMOP ADM PARQUE': 'EPMMOP', 'EPMMOP GUARDIAS': 'EPMMOP',
    'JEFATURA SC': 'Jefatura SC', 'AZONAL': 'Adm. Zonal', 'INTENDENCIA': 'Intendencia',
    'TENENCIA POL TICA': 'Tenencia Política', 'S INCLUSI N': 'Sec. Inclusión',
    'PATRONATO': 'Patronato', 'EP RASTRO': 'EP Rastro', 'MIGRACIÓN (MDI)': 'Migración', 'BUS DE CBQ': 'CBQ',
    'UNIDADES TÁCTICAS PPNN (SEGÚN PLANIFICACIÓN INTERNA)': 'PPNN', 'COS': 'COSP',
    'DINAPEN': 'PPNN', 'DIR. TRATA PERSONAS': 'PPNN', 'UNIDAD DE TRATA DE PERSONAS': 'PPNN',
}
def canon_inst(k):
    k = re.sub(r'\s+', ' ', k).strip(' ,')
    if not k:
        return 'Otras'
    if '(CACMQ)' in k: return 'CACMQ'
    if '( AMT)' in k or '(AMT)' in k: return 'AMT'
    return INST_MAP.get(k, k)

def parse_inst(s):
    s = (s or '').replace('\n', ' ').strip()
    out = {}
    for m in re.finditer(r'([A-ZÁÉÍÓÚÑa-z\.\s/]+?(?:\s*\([A-Za-zÁÉÍÓÚÑ\s\.]+\))?)\s*\((\d+)\)', s):
        k = canon_inst(m.group(1))
        out[k] = out.get(k, 0) + int(m.group(2))
    return out

In [ ]:
# Lectura del reporte Macro (resultados)
macro_rows, rep_macro = [], None
with pdfplumber.open(MACRO_PDF) as pdf:
    txt0 = pdf.pages[0].extract_text() or ''
    m_ = re.search(r'FECHA DE REPORTE:\s*(.+?)\s{2,}', txt0)
    for p in pdf.pages:
        for t in p.extract_tables():
            for r in t:
                if r and r[0] and r[0] != 'FECHA' and len(r) == 9:
                    macro_rows.append([(c or '').strip() for c in r])
rep_macro = "4 de noviembre de 2026, 03:04 p. m."
print("Filas brutas Macro:", len(macro_rows))

# Responsables: unificar variantes (mayúsculas / tildes) usando la variante con tilde más frecuente
cnt = collections.Counter(r[3] for r in macro_rows)
best = {}
for name, c in cnt.items():
    key = strip_acc(name).lower()
    sc = (any(ch in name for ch in 'áéíóúÁÉÍÓÚ'), name != name.upper(), c)
    if key not in best or sc > best[key][0]:
        best[key] = (sc, name)
def canon_resp(n):
    return 'Sin dato' if n in ('-', '', '—') else best[strip_acc(n).lower()][1]

macro = []
for i, r in enumerate(macro_rows):
    fecha, tipo, horario, resp, az, parr, estado, inst, res = r
    tot = re.search(r'Total\s+(\d+)', inst)
    d = {
        'id': i + 1,
        'f': datetime.datetime.strptime(fecha, '%d/%m/%Y').strftime('%Y-%m-%d'),
        'tipo': tipo, 'horario': horario, 'resp': canon_resp(resp),
        'az': az.replace('AZ. ', ''), 'parroquia': parr, 'estado': estado,
        'total': int(tot[1]) if tot else None,
        'inst': parse_inst(re.sub(r'Total\s+\d+', '', inst)),
        'horas': dur(horario), 'h0': start_hour(horario),
        'resultados_reportados': res.strip() not in ('—', ''),
    }
    for lab, key in [('Est', 'est'), ('Cacheos', 'cacheos'), ('Armas', 'armas'),
                     ('Sanciones', 'sanciones'), ('Alcohol', 'alcohol'), ('Cajetillas', 'cajetillas')]:
        mm = re.search(lab + r'\.?:\s*(\d+)', res)
        d[key] = int(mm[1]) if mm else 0
    # Operativo analítico: ejecutado, con personal y con resultados reportados
    d['valido'] = (estado == 'EJECUTADO' and d['total'] is not None and d['total'] > 0 and d['resultados_reportados'])
    macro.append(d)

In [ ]:
# Tipo de participante (clasificación propia de las instituciones)
# Criterio: agrupa las instituciones que aparecen en el reporte según su rol en el operativo.
# Las siglas que no se pueden asignar con certeza (UBA, AMCC, OTRA) quedan en "Otros".
TIPOS_PART = ['Fuerza pública', 'Control municipal', 'Gestión de riesgos y emergencias', 'Gobierno nacional', 'Otros']
INST_TIPO = {
    'PPNN': 'Fuerza pública', 'FFAA': 'Fuerza pública',
    'CACMQ': 'Control municipal', 'AMT': 'Control municipal', 'AMC': 'Control municipal',
    'Comisaría': 'Control municipal', 'Quito Turismo': 'Control municipal',
    'CBQ': 'Gestión de riesgos y emergencias', 'SGSCGR': 'Gestión de riesgos y emergencias',
    'JZSCGR': 'Gestión de riesgos y emergencias', 'COE': 'Gestión de riesgos y emergencias',
    'Intendencia': 'Gobierno nacional', 'Min. Interior': 'Gobierno nacional',
}
def tipo_de(k): return INST_TIPO.get(k, 'Otros')
for d in macro:
    p = {}
    for k, n in d['inst'].items():
        p[tipo_de(k)] = p.get(tipo_de(k), 0) + n
    d['part'] = p

# Verificación interna: total declarado = suma por institución
dif = [(d['f'], d['total'], sum(d['inst'].values())) for d in macro if d['total'] and d['total'] != sum(d['inst'].values())]
assert not dif, dif
dup = collections.Counter((d['f'], d['parroquia'], d['tipo'], d['horario']) for d in macro)
print("Duplicados exactos (fecha, parroquia, tipo, horario):", sum(1 for v in dup.values() if v > 1))

In [ ]:
# Catálogos
metricas = [
    {'k': 'sanciones', 'label': 'Sanciones', 'nota': 'Sanciones impuestas'},
    {'k': 'cacheos', 'label': 'Cacheos', 'nota': 'Personas cacheadas'},
    {'k': 'est', 'label': 'Establecimientos (Est.)', 'nota': 'Establecimientos controlados'},
    {'k': 'armas', 'label': 'Armas', 'nota': 'Armas retenidas'},
    {'k': 'alcohol', 'label': 'Alcohol', 'nota': 'Alcohol retenido (unidades según reporte)'},
    {'k': 'cajetillas', 'label': 'Cajetillas', 'nota': 'Cajetillas de cigarrillos retenidas'},
]
insts = sorted({k for d in macro for k in d['inst']})
meses = sorted({d['f'][:7] for d in macro})
azs = sorted({d['az'] for d in macro})

In [ ]:
# Conciliación y resumen verificable
val = [d for d in macro if d['valido']]
def suma(rows, k): return sum(r[k] for r in rows)
resumen = {
    'macro_filas': len(macro),
    'macro_ejecutados': sum(1 for d in macro if d['estado'] == 'EJECUTADO'),
    'macro_suspendidos': sum(1 for d in macro if d['estado'] == 'SUSPENDIDO'),
    'macro_validos': len(val),
    'macro_sin_resultados': sum(1 for d in macro if d['estado'] == 'EJECUTADO' and not d['valido']),
    'personal_valido': suma(val, 'total'),
    **{f'tot_{m["k"]}': suma(val, m['k']) for m in metricas},
    'personal_por_tipo': {t: sum(d['part'].get(t, 0) for d in val) for t in TIPOS_PART},
}
for k in ('sanciones', 'cacheos', 'est', 'armas'):
    resumen[f'por100_{k}'] = 100 * resumen[f'tot_{k}'] / resumen['personal_valido']
print(json.dumps(resumen, indent=1, ensure_ascii=False))

In [ ]:
# APP_DATA
A = {
    'meta': {
        'titulo': 'Productividad de operativos macro — Secretaría de Seguridad',
        'reporte_macro': rep_macro,
        'macro_desde': min(d['f'] for d in macro), 'macro_hasta': max(d['f'] for d in macro),
        'fuente': 'SIGES — Unidad de Políticas y Planeamiento',
    },
    'catalogos': {'meses': meses, 'az': azs, 'metricas': metricas,
                  'tipos_macro': sorted({d['tipo'] for d in macro}),
                  'tipos_part': TIPOS_PART, 'inst_tipo': {k: tipo_de(k) for k in insts}},
    'resumen': resumen,
    'macro': macro,
}
print("APP_DATA listo:", len(json.dumps(A, ensure_ascii=False)) // 1024, "KB")

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "261106_operativos_macro_v2.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")